# Определение тональности отзывов на банки с помощью классических алгоритмов машинного обучения

Используем логистическую регрессию и мешок слов.

Чтобы запускать и редактировать код, сохраните копию этого ноутбука себе (Файл -> Создать копию на Диске). Свою копию вы сможете изменять и запускать.

Учебный курс "[Программирование глубоких нейронных сетей на Python](https://openedu.ru/course/urfu/PYDNN/)".

<a target="_blank" href="https://colab.research.google.com/github/sozykin/dlpython_course/blob/master/text_processing/text_classification.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



In [ ]:
pip install pymorphy3[fast]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import numpy as np
import pymorphy3
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from collections import Counter
from pathlib import Path

In [7]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sozyk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\sozyk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sozyk\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

Константы

In [8]:
max_words = 10000
random_state = 42

## Загружаем и готовим набор данных

In [9]:
data_path = 'data'

In [10]:
path = Path(data_path)

if not path.exists():
    path.mkdir(parents=True, exist_ok=True)

In [11]:
!curl -s -L -o data/banks.csv "https://www.dropbox.com/scl/fi/mhiwx4plaua183bnuqt5e/train2.csv?rlkey=wv4dalbsutmzu5br9qv5ez21v&dl=1"

In [13]:
banks = pd.read_csv('data/banks.csv', index_col='id');

In [14]:
banks

,text,label
id,,
0,В Альфа-Банке работает замечательная девушка -...,1
1,Оформляя рассрочку в м. Видео в меге тёплый ст...,0
2,Очень порадовала оперативность работы в банке....,1
3,Имела неосторожность оформить потреб. кредит в...,0
4,Небольшая предыстория: Нашел на сайте MDM банк...,0
...,...,...
13994,"О высокой надёжности МКБ, порядочности и добро...",1
13995,"Обслуживаюсь в офисе на Чернореченской 42а, ка...",1
13996,Попала сегодня в очень неприятную ситуацию. Ре...,1


In [15]:
def preprocess(text, stop_words, punctuation_marks, morph):
    tokens = word_tokenize(text.lower())
    preprocessed_text = []
    for token in tokens:
        if token not in punctuation_marks:
            lemma = morph.parse(token)[0].normal_form
            if lemma not in stop_words:
                preprocessed_text.append(lemma)
    return preprocessed_text

In [16]:
punctuation_marks = ['!', ',', '(', ')', ':', '-', '?', '.', '..', '...', '«', '»', ';', '–', '--']
stop_words = stopwords.words("russian")
morph = pymorphy3.MorphAnalyzer()

In [17]:
banks['Preprocessed_texts'] = banks.apply(lambda row: preprocess(row['text'], stop_words, punctuation_marks, morph), axis=1)

In [18]:
banks

,text,label,Preprocessed_texts
id,,,
0,В Альфа-Банке работает замечательная девушка -...,1,"[альфа-банк, работать, замечательный, девушка,..."
1,Оформляя рассрочку в м. Видео в меге тёплый ст...,0,"[оформлять, рассрочка, м., видео, мег, тёплый,..."
2,Очень порадовала оперативность работы в банке....,1,"[очень, порадовать, оперативность, работа, бан..."
3,Имела неосторожность оформить потреб. кредит в...,0,"[иметь, неосторожность, оформить, потреба, кре..."
4,Небольшая предыстория: Нашел на сайте MDM банк...,0,"[небольшой, предыстория, найти, сайт, mdm, бан..."
...,...,...,...
13994,"О высокой надёжности МКБ, порядочности и добро...",1,"[высокий, надёжность, мкб, порядочность, добро..."
13995,"Обслуживаюсь в офисе на Чернореченской 42а, ка...",1,"[обслуживаться, офис, чернореченский, 42а, физ..."
13996,Попала сегодня в очень неприятную ситуацию. Ре...,1,"[попасть, сегодня, очень, неприятный, ситуация..."


Считаем частоту слов во всех отзывах

In [19]:
words = Counter()

In [20]:
for txt in banks['Preprocessed_texts']:
    words.update(txt)

In [21]:
words.most_common(20)

[('банк', 58189),
 ('карта', 30560),
 ('это', 28660),
 ('всё', 20599),
 ('день', 15729),
 ('сотрудник', 14527),
 ('который', 14519),
 ('кредит', 14490),
 ('деньга', 13701),
 ('счёт', 13681),
 ('отделение', 13414),
 ('клиент', 12681),
 ('мочь', 11140),
 ('свой', 11128),
 ('год', 10378),
 ('сказать', 9990),
 ('вопрос', 9638),
 ('ещё', 9606),
 ('очень', 9226),
 ('весь', 9004)]

Создаем словарь, упорядоченный по частоте

В словаре будем использовать 2 специальных кода:
- Код заполнитель: 0
- Неизвестное слово: 1

Нумерация слов в словаре начинается с 2.

In [22]:
# Словарь, отображающий слова в коды
word_to_index = dict()
# Словарь, отображающий коды в слова
index_to_word = dict()

Создаем словари

In [23]:
for i, word in enumerate(words.most_common(max_words - 2)):
    word_to_index[word[0]] = i + 2
    index_to_word[i + 2] = word[0]

In [24]:
word_to_index

{'банк': 2,
 'карта': 3,
 'это': 4,
 'всё': 5,
 'день': 6,
 'сотрудник': 7,
 'который': 8,
 'кредит': 9,
 'деньга': 10,
 'счёт': 11,
 'отделение': 12,
 'клиент': 13,
 'мочь': 14,
 'свой': 15,
 'год': 16,
 'сказать': 17,
 'вопрос': 18,
 'ещё': 19,
 'очень': 20,
 'весь': 21,
 'время': 22,
 'сумма': 23,
 'кредитный': 24,
 'получить': 25,
 'офис': 26,
 'проблема': 27,
 'заявление': 28,
 'договор': 29,
 'работа': 30,
 'платёж': 31,
 'банкомат': 32,
 'телефон': 33,
 'позвонить': 34,
 'месяц': 35,
 'документ': 36,
 'дать': 37,
 'ответ': 38,
 'решить': 39,
 'хотеть': 40,
 'обслуживание': 41,
 'звонить': 42,
 'ваш': 43,
 'работать': 44,
 'услуга': 45,
 'претензия': 46,
 'прийти': 47,
 'вклад': 48,
 'звонок': 49,
 'номер': 50,
 'написать': 51,
 'большой': 52,
 'ситуация': 53,
 'рубль': 54,
 'человек': 55,
 'минута': 56,
 'сделать': 57,
 'просто': 58,
 'говорить': 59,
 'средство': 60,
 'альфа-банк': 61,
 'заявка': 62,
 'срок': 63,
 'очередь': 64,
 '2': 65,
 'первый': 66,
 'знать': 67,
 'информаци

Функция для преобразования списка слов в список кодов

In [25]:
def text_to_sequence(txt, word_to_index):
    seq = []
    for word in txt:
        index = word_to_index.get(word, 1) # 1 означает неизвестное слово
        # Неизвестные слова не добавляем в выходную последовательность
        if index != 1:
            seq.append(index)
    return seq

Преобразуем все тексты в последовательность кодов слов

In [26]:
banks['Sequences'] = banks.apply(lambda row: text_to_sequence(row['Preprocessed_texts'], word_to_index), axis=1)

In [27]:
banks

,text,label,Preprocessed_texts,Sequences
id,,,,
0,В Альфа-Банке работает замечательная девушка -...,1,"[альфа-банк, работать, замечательный, девушка,...","[61, 44, 896, 76, 194, 1842, 344, 2685, 396, 1..."
1,Оформляя рассрочку в м. Видео в меге тёплый ст...,0,"[оформлять, рассрочка, м., видео, мег, тёплый,...","[254, 836, 1155, 3189, 3865, 2956, 7316, 163, ..."
2,Очень порадовала оперативность работы в банке....,1,"[очень, порадовать, оперативность, работа, бан...","[20, 1033, 890, 30, 2, 461, 175, 3, 613, 1743,..."
3,Имела неосторожность оформить потреб. кредит в...,0,"[иметь, неосторожность, оформить, потреба, кре...","[113, 4790, 70, 2343, 9, 61, 20, 2639, 1264, 3..."
4,Небольшая предыстория: Нашел на сайте MDM банк...,0,"[небольшой, предыстория, найти, сайт, mdm, бан...","[405, 3824, 262, 85, 2, 633, 3, 4866, 2, 283, ..."
...,...,...,...,...
13994,"О высокой надёжности МКБ, порядочности и добро...",1,"[высокий, надёжность, мкб, порядочность, добро...","[388, 2296, 792, 5000, 8128, 7, 3073, 120, 427..."
13995,"Обслуживаюсь в офисе на Чернореченской 42а, ка...",1,"[обслуживаться, офис, чернореченский, 42а, физ...","[349, 26, 9465, 2126, 191, 16, 26, 134, 64, 11..."
13996,Попала сегодня в очень неприятную ситуацию. Ре...,1,"[попасть, сегодня, очень, неприятный, ситуация...","[502, 104, 20, 971, 53, 39, 213, 221, 3, 47, 3..."


## Готовим данные для обучения

### Выделяем данные для обучения и тестирования

In [28]:
train, test = train_test_split(banks, test_size=0.2)

In [29]:
train

,text,label,Preprocessed_texts,Sequences
id,,,,
10648,Около месяца назад на мой мобильный телефон ст...,0,"[около, месяц, назад, мобильный, телефон, стат...","[190, 35, 200, 292, 33, 91, 146, 341, 2, 249, ..."
2922,У меня имеется открытый вклад Великолепная сем...,0,"[иметься, открытый, вклад, великолепный, семёр...","[621, 1028, 48, 2561, 7301, 1043, 171, 60, 579..."
8955,"Итак, в итоге решил я перейти из Райфайзена в ...",1,"[итак, итог, решить, перейти, райфайзть, аванг...","[1537, 128, 39, 1321, 249, 1642, 249, 792, 166..."
10410,В сентябре был открыт валютный счет-копилка в ...,0,"[сентябрь, открыть, валютный, счёт-копилка, эт...","[534, 115, 771, 4, 2, 806, 2215, 60, 7903, 494..."
11095,Давно являюсь клиентом Банка и это давно не су...,1,"[давно, являться, клиент, банк, это, давно, су...","[384, 84, 13, 2, 4, 384, 1197, 4, 2, 2399, 210..."
...,...,...,...,...
4172,Добрый день!Хотел бы поблагодарить отдел прете...,1,"[добрый, день, хотеть, поблагодарить, отдел, п...","[173, 6, 40, 761, 252, 1778, 30, 1274, 960, 11..."
11170,Пользуюсь кредиткой Авангарда с 2006 года. Нра...,1,"[пользоваться, кредитка, авангард, 2006, год, ...","[75, 360, 249, 1757, 16, 575, 5, 81, 225, 167,..."
8446,Имел возможность сравнить качество обслуживани...,1,"[иметь, возможность, сравнить, качество, обслу...","[113, 167, 1976, 320, 41, 2, 3466, 187, 1741, ..."


In [30]:
test

,text,label,Preprocessed_texts,Sequences
id,,,,
13777,У меня оформлен автокредит в Сетелем Банке - п...,1,"[оформить, автокредит, сетель, банк, причина, ...","[70, 714, 538, 2, 224, 588, 77, 211, 2059, 465..."
10115,Давно не пользовался услугами Сбербанка. Очень...,1,"[давно, пользоваться, услуга, сбербанк, очень,...","[384, 75, 45, 136, 20, 384, 1738, 351, 159, 14..."
10842,"Я - несчастный бывший клиент Пробизнесбанка, п...",0,"[несчастный, бывший, клиент, пробизнесбанк, пр...","[3004, 1489, 13, 8670, 7424, 8, 91, 273, 48, 4..."
8090,История такая! Прорекламировали мне карту Плат...,0,"[история, прорекламировать, карта, платиновый,...","[161, 3, 2117, 3, 2141, 20, 3344, 311, 423, 25..."
9749,По инициативе банка были заблокированы 2 карты...,0,"[инициатива, банк, заблокировать, 2, карта, де...","[2075, 2, 365, 65, 3, 205, 24, 1962, 880, 224,..."
...,...,...,...,...
13852,Добрый день! Обычно люди склонны жаловаться на...,1,"[добрый, день, обычно, человек, склонный, жало...","[173, 6, 612, 55, 1733, 556, 3209, 88, 164, 19..."
1176,"Брал кредит, всё нормально выплатил, единствен...",1,"[брать, кредит, всё, нормально, выплатить, еди...","[162, 9, 5, 763, 1085, 429, 1845, 24, 3, 463, ..."
7148,Несколько лет назад мы с дочерью делали ремонт...,1,"[несколько, год, назад, дочь, делать, ремонт, ...","[80, 16, 200, 2304, 132, 1657, 310, 39, 895, 5..."


### Разделяем метки классов и данные для обучения

Данные для обучения

In [31]:
x_train_seq = train['Sequences']
y_train = train['label']

In [32]:
x_train_seq

id
10648    [190, 35, 200, 292, 33, 91, 146, 341, 2, 249, ...
2922     [621, 1028, 48, 2561, 7301, 1043, 171, 60, 579...
8955     [1537, 128, 39, 1321, 249, 1642, 249, 792, 166...
10410    [534, 115, 771, 4, 2, 806, 2215, 60, 7903, 494...
11095    [384, 84, 13, 2, 4, 384, 1197, 4, 2, 2399, 210...
                               ...                        
4172     [173, 6, 40, 761, 252, 1778, 30, 1274, 960, 11...
11170    [75, 360, 249, 1757, 16, 575, 5, 81, 225, 167,...
8446     [113, 167, 1976, 320, 41, 2, 3466, 187, 1741, ...
12281    [283, 36, 272, 1057, 169, 789, 1108, 102, 900,...
1300     [486, 273, 1489, 13, 2, 380, 13, 232, 5, 90, 3...
Name: Sequences, Length: 11199, dtype: object

In [33]:
y_train

id
10648    0
2922     0
8955     1
10410    0
11095    1
        ..
4172     1
11170    1
8446     1
12281    0
1300     0
Name: label, Length: 11199, dtype: int64

Данные для тестирования

In [34]:
x_test_seq = test['Sequences']
y_test = test['label']

In [35]:
x_test_seq

id
13777    [70, 714, 538, 2, 224, 588, 77, 211, 2059, 465...
10115    [384, 75, 45, 136, 20, 384, 1738, 351, 159, 14...
10842    [3004, 1489, 13, 8670, 7424, 8, 91, 273, 48, 4...
8090     [161, 3, 2117, 3, 2141, 20, 3344, 311, 423, 25...
9749     [2075, 2, 365, 65, 3, 205, 24, 1962, 880, 224,...
                               ...                        
13852    [173, 6, 612, 55, 1733, 556, 3209, 88, 164, 19...
1176     [162, 9, 5, 763, 1085, 429, 1845, 24, 3, 463, ...
7148     [80, 16, 200, 2304, 132, 1657, 310, 39, 895, 5...
895      [1179, 15, 2544, 299, 270, 7, 136, 554, 5818, ...
3434     [1702, 1212, 7, 61, 907, 872, 300, 2605, 122, ...
Name: Sequences, Length: 2800, dtype: object

In [36]:
y_test

id
13777    1
10115    1
10842    0
8090     0
9749     0
        ..
13852    1
1176     1
7148     1
895      1
3434     1
Name: label, Length: 2800, dtype: int64

## Создаем мешок слов

In [37]:
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        for index in sequence:
            results[i, index] += 1.
    return results

In [38]:
x_train = vectorize_sequences(x_train_seq, max_words)

In [39]:
x_test = vectorize_sequences(x_test_seq, max_words)

In [40]:
x_train[0][:100]

array([0., 0., 4., 0., 2., 0., 1., 1., 2., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
       1., 1., 0., 0., 0., 0., 0., 0., 2., 0., 0., 0., 0., 0., 0., 0., 3.,
       0., 0., 0., 0., 2., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 3., 0., 1., 0., 0., 0., 0., 3.,
       0., 0., 0., 0., 0., 0., 2., 0., 0., 0., 0., 0., 0., 0., 0.])

In [41]:
len(x_train[0])

10000

## Создаем модель машинного обучения

In [42]:
lr = LogisticRegression(random_state=random_state, max_iter=500)

## Обучаем модель машинного обучения

In [43]:
lr.fit(x_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,500
,multi_class,'deprecated'


## Оцениваем качество обучения на тестовом наборе данных

Определяем долю правильных ответов (accuracy) на тестовом наборе данных

In [44]:
lr.score(x_test, y_test)

0.9414285714285714

## Применяем модель для определения тональности отзыва на банк

**Позитивный отзыв**

In [45]:
positive_text = """Брал кредит в Мегабанке на автомобиль. Выдали за один день. Никаких скрытых комиссий и переплат.
У банка удобное мобильное приложение, через которое можно быстро отправить ежемесячный платеж.
Досрочное гасить начал через три месяца. Я доволен оперативностью и удобством. Огромное спасибо!
"""

Подготовка текста к обработке

In [46]:
positive_preprocessed_text = preprocess(positive_text, stop_words, punctuation_marks, morph)

In [47]:
positive_preprocessed_text

['брать',
 'кредит',
 'мегабанк',
 'автомобиль',
 'выдать',
 'день',
 'никакой',
 'скрытый',
 'комиссия',
 'переплата',
 'банк',
 'удобный',
 'мобильный',
 'приложение',
 'который',
 'быстро',
 'отправить',
 'ежемесячный',
 'платёж',
 'досрочный',
 'гасить',
 'начать',
 'месяц',
 'довольный',
 'оперативность',
 'удобство',
 'огромный',
 'спасибо']

In [48]:
positive_seq = text_to_sequence(positive_preprocessed_text, word_to_index)

In [49]:
positive_seq

[162,
 9,
 942,
 129,
 6,
 79,
 1866,
 78,
 1007,
 2,
 274,
 292,
 747,
 8,
 122,
 195,
 475,
 31,
 345,
 1399,
 266,
 35,
 286,
 890,
 1293,
 299,
 73]

In [50]:
positive_bow = vectorize_sequences([positive_seq], max_words)

In [51]:
positive_bow[0][0:100]

array([0., 0., 1., 0., 0., 0., 1., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

Выполняем распознавание

In [52]:
result = lr.predict(positive_bow)

In [53]:
result

array([1])

**Негативный отзыв**

In [54]:
negative_text = """Взял кредит в ТакСебеБанке на автомобиль. В договор включили обязательный контракт
на помощь на дороге, который мне не нужен. Узнал об этом только во время подписания договора, иначе бы отказался.
Альтернативы была страхование жизни, но мне это даже не предложили. Скорее всего, менеджер продвигает
продажи услуг этой компании в ущерб интересов клиента. Как минимум, непорядочно и непрофессионально.
У банка ужасное мобильное приложение, из-за которого с меня взяли штраф 10 тыс.руб. По требованиям
банка после покупки автомобиля в приложении нужно загрузить ПТС. Я загрузил и проверил, что ПТС в приложении есть.
Но через некоторое время ПТС из приложения пропал и с меня взяли штраф. Никому не рекомендую связываться с ТакСебеБанком.
"""

In [55]:
negative_preprocessed_text = preprocess(negative_text, stop_words, punctuation_marks, morph)
negative_seq = text_to_sequence(negative_preprocessed_text, word_to_index)
negative_bow = vectorize_sequences([negative_seq], max_words)

In [56]:
negative_bow[0][0:100]

array([0., 0., 2., 0., 2., 0., 0., 0., 2., 1., 0., 0., 0., 1., 0., 0., 0.,
       0., 0., 0., 0., 1., 2., 0., 0., 0., 0., 0., 0., 2., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.])

In [57]:
result = lr.predict(negative_bow)

In [58]:
result

array([0])

In [59]:
result = lr.predict_proba(negative_bow)

In [60]:
result

array([[0.99678078, 0.00321922]])